In [0]:
df = spark.table("olist_ecom.bronze.bronze_orders")
df.printSchema()
df.show(5, truncate=False)

In [0]:
#profile the status column
df.groupBy("order_status").count().orderBy("count", ascending=False).show()

In [0]:
#count nulls per column:
from pyspark.sql.functions import col, sum as spark_sum, when

df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
from pyspark.sql.functions import to_timestamp, col

ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

df = spark.table("olist_ecom.bronze.bronze_orders")

for c in ts_cols:
    df = df.withColumn(c, to_timestamp(col(c), "yyyy-MM-dd HH:mm:ss"))

df.write.format("delta").mode("overwrite") \
  .saveAsTable("olist_ecom.silver.silver_orders")

In [0]:
#Then verify -- first check
from pyspark.sql.functions import sum as spark_sum, when

s = spark.table("olist_ecom.silver.silver_orders")
s.printSchema()
s.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ts_cols
]).show()

In [0]:
#second check
s.filter(col("order_delivered_customer_date") < col("order_purchase_timestamp")).count()